# Practice 3 (3.10)

### Imports

In [1]:
import pandas as pd
import featuretools as ft
import numpy as np
from datetime import datetime, timedelta

ModuleNotFoundError: No module named 'pkg_resources'

### Locking seeds

In [ ]:
np.random.seed(42)

### Creating data

In [ ]:
customers = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'join_date': pd.to_datetime([
        '2023-01-15', '2023-03-22', '2023-05-10', '2023-07-01', '2023-09-18'
    ]),
    'region': ['North', 'South', 'East', 'West', 'North']
})

In [ ]:
# --- products ---
products = pd.DataFrame({
    'product_id': [101, 102, 103, 104],
    'category': ['Electronics', 'Clothing', 'Food', 'Electronics'],
    'price': [500.0, 50.0, 10.0, 300.0]
})

In [ ]:
orders_data = []
order_id = 1000
for client_id in customers['client_id']:
    n_orders = np.random.randint(1, 5)
    for _ in range(n_orders):
        order_id += 1
        order_date = pd.Timestamp('2023-01-01') + pd.to_timedelta(
            np.random.randint(0, 365), unit='D'
        )
        orders_data.append({
            'order_id': order_id,
            'client_id': client_id,
            'order_date': order_date,
            'total_amount': round(np.random.uniform(20, 2000), 2)
        })
orders = pd.DataFrame(orders_data)

In [ ]:
items_data = []
item_id = 1
for _, order in orders.iterrows():
    n_items = np.random.randint(1, 4)
    for _ in range(n_items):
        product = products.sample(1).iloc[0]
        items_data.append({
            'item_id': item_id,
            'order_id': order['order_id'],
            'product_id': product['product_id'],
            'quantity': np.random.randint(1, 10)
        })
        item_id += 1
order_items = pd.DataFrame(items_data)

In [ ]:
print("=== Таблицы ===")
print("customers:\n", customers.head(), "\n")
print("orders:\n", orders.head(), "\n")
print("products:\n", products.head(), "\n")
print("order_items:\n", order_items.head(), "\n")

### Creating EntitySet

In [ ]:
es = ft.EntitySet(id='retail')

es = es.add_dataframe(
    dataframe_name='customers',
    dataframe=customers,
    index='client_id',
    time_index='join_date'
)

es = es.add_dataframe(
    dataframe_name='orders',
    dataframe=orders,
    index='order_id',
    time_index='order_date'
)

es = es.add_dataframe(
    dataframe_name='products',
    dataframe=products,
    index='product_id'
)

es = es.add_dataframe(
    dataframe_name='order_items',
    dataframe=order_items,
    index='item_id'
)

### Creating relationships

In [ ]:
es = es.add_relationship('customers', 'client_id', 'orders', 'client_id')
es = es.add_relationship('orders', 'order_id', 'order_items', 'order_id')
es = es.add_relationship('products', 'product_id', 'order_items', 'product_id')

print("=== EntitySet ===")
print(es)

### DFS

In [ ]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    agg_primitives=['sum', 'mean', 'count', 'max', 'min', 'std'],
    trans_primitives=['day', 'month', 'year', 'weekday'],
    max_depth=2,
    verbose=True
)

In [ ]:
print("\n=== Размер матрицы признаков ===")
print(f"Shape: {feature_matrix.shape}")
print(f"Строк: {feature_matrix.shape[0]}, Признаков: {feature_matrix.shape[1]}")

print("\n=== Список сгенерированных признаков ===")
for i, f in enumerate(feature_defs, 1):
    print(f"{i:3d}. {f.get_name()}")

print("\n=== Первые строки матрицы признаков ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(feature_matrix.head())